# Allianz: Predicción de Débito Directo con Machine Learning

**Rol:** Equipo de Análisis de Datos — Allianz Data Office  
**Tipo de proyecto:** Clasificación Supervisada con Machine Learning  
**Objetivo de negocio:** Predecir el uso de débito directo e identificar segmentos prioritarios para campañas de adopción.

---

## Resumen Ejecutivo

Allianz Benelux tiene una tasa de adopción de débito directo significativamente menor (~21%) en comparación con el promedio del mercado (~65%). Los pagos por débito directo mejoran la predictibilidad del flujo de caja, reducen la carga administrativa y disminuyen el riesgo de cobros atrasados o incompletos.

Este notebook presenta un **proyecto de clasificación binaria supervisada** desarrollado por el equipo de Allianz Data Office. Nuestro objetivo es predecir, con base en las características del contrato, cliente, corredor y producto, qué contratos tienen probabilidad de usar débito directo. El modelo permitirá a Allianz priorizar una campaña dirigida en lugar de contactar a todos los clientes, haciendo el esfuerzo de adopción más eficiente y rentable.

Seguimos la metodología **CRISP-DM** (Proceso Estándar Entre Industrias para Minería de Datos) en seis fases:
1. Comprensión del Negocio
2. Comprensión de los Datos
3. Preparación de los Datos
4. Modelado
5. Evaluación
6. Despliegue / Recomendaciones de Negocio

Entrenamos y comparamos cinco modelos de clasificación: Árbol de Decisión, Random Forest, Regresión Logística, Máquina de Soporte Vectorial y XGBoost. El mejor modelo será recomendado para uso operativo.

---

## Tabla de Contenido
1. Comprensión del Negocio
2. Diccionario de Datos y Contexto del Caso
3. Importar Librerías
4. Cargar Dataset
5. Inspección Inicial de Datos
6. Análisis Exploratorio de Datos (EDA)
7. Limpieza de Datos
8. Ingeniería de Características
9. Pipeline de Preprocesamiento
10. División Entrenamiento-Prueba
11. Revisión del Desbalance de Clases
12. Modelo 1: Árbol de Decisión
13. Modelo 2: Random Forest
14. Modelo 3: Regresión Logística
15. Modelo 4: Máquina de Soporte Vectorial
16. Modelo 5: XGBoost
17. Comparación de Modelos
18. Matrices de Confusión
19. ROC-AUC y Curvas ROC
20. Importancia de Características e Interpretación
21. Selección del Modelo Final
22. Recomendaciones de Negocio
23. Limitaciones y Próximos Pasos
24. Conclusiones Finales
25. Guardar Modelo con Joblib


---
## 1. Comprensión del Negocio

### 1.1 Contexto de la Empresa

Allianz es una de las empresas líderes mundiales en seguros y servicios financieros. En Bélgica, Allianz opera a través de una red de corredores, ofreciendo una amplia gama de productos de seguros de propiedad, responsabilidad civil, automóvil, accidentes e ingeniería.

### 1.2 El Problema del Débito Directo

**¿Qué es el débito directo?** El débito directo es un método de pago automático donde Allianz cobra las primas de seguro directamente de la cuenta bancaria del cliente en la fecha de vencimiento, sin requerir que el cliente inicie cada pago manualmente.

**¿Por qué importa?**
- Mejora la **predictibilidad de los flujos de caja** para Allianz
- Reduce los **costos de cobro manual** y la carga administrativa
- Reduce los **pagos de primas atrasados o incompletos**
- Acorta el **período de cuentas por cobrar** dentro del ciclo de conversión de efectivo
- Mejora la **gestión del capital de trabajo**
- Proporciona una mejor **experiencia al cliente** mediante pagos automáticos y puntuales

**La brecha:** El promedio del mercado para la adopción de débito directo en seguros es aproximadamente **65%**. La tasa de adopción actual de Allianz es solo **~21%**. Esto representa una brecha operativa y financiera significativa.

### 1.3 Pregunta de Negocio

> *¿Qué características de clientes, corredores, productos y contratos están asociadas con el uso de débito directo, y cómo puede Allianz usar un modelo predictivo para priorizar futuras campañas de adopción de débito directo?*

### 1.4 Enfoque de Machine Learning

Planteamos esto como un **problema de clasificación binaria supervisada**:
- **Variable objetivo:** `Is_direct_debit`
- **Clase 1:** El contrato usa débito directo
- **Clase 0:** El contrato no usa débito directo

Entrenaremos múltiples modelos de clasificación, compararemos su desempeño y seleccionaremos el modelo que mejor equilibre precisión predictiva, recall para la clase positiva e interpretabilidad de negocio.


---
## 2. Diccionario de Datos y Contexto del Caso

El dataset está a nivel de **contrato** — cada fila representa un contrato de seguro. Un cliente puede tener múltiples contratos.

| Columna | Tipo | Significado de Negocio |
|---|---|---|
| `Broker_account_number` | categórico | Identificador del corredor — excluido del modelado (ID) |
| `Contract_number` | categórico | Identificador del contrato — excluido del modelado (ID) |
| `Customer_segment` | categórico | Retail, PyME o Mediana Corporación |
| `Line_of_business` | categórico | Línea de seguro (Automóvil, Propiedad, Responsabilidad, etc.) |
| `Product_type` | categórico | Código de producto específico dentro de cada línea |
| `Annual_premium` | float | Prima anual cobrada por el contrato |
| `Payment_frequency` | categórico | Mensual, Trimestral, Semestral, Anual |
| `Customer_ID` | entero | Identificador del cliente — excluido del modelado (ID) |
| `Customer_age` | categórico | Grupo de edad; 'No age' = cliente empresarial |
| `Customer_type` | categórico | Persona física o Empresa |
| `Customer_region` | categórico | Región belga (BRU, WAL, FLA) |
| `Customer_province` | categórico | Provincia belga |
| `Broker_region` | categórico | Región belga del corredor |
| `Broker_province` | categórico | Provincia belga del corredor |
| `Is_direct_debit` | entero | **Objetivo:** 0 = sin débito directo, 1 = débito directo |
| `Broker_cor` | float | Ratio combinado del corredor; más bajo = más rentable |
| `Customer_urbanization` | categórico | Urbano o Rural (ubicación del cliente) |
| `Broker_urbanization` | categórico | Urbano o Rural (ubicación del corredor) |

### Nota Importante sobre la Frecuencia de Pago (Posible Fuga de Datos)

Las reglas de negocio de Allianz **obligan a los contratos de pago mensual a usar débito directo**. Esto significa que `Payment_frequency = Monthly` es un predictor casi perfecto del débito directo. Por ello:
- **Estrategia A:** Incluir `Payment_frequency` y discutir el riesgo de fuga.
- **Estrategia B:** Entrenar un modelo alternativo excluyendo `Payment_frequency` para entender los factores conductuales más profundos.


---
## 3. Importar Librerías


In [ ]:
# Manipulación de datos y cómputo numérico
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Utilidades de preprocesamiento y pipeline
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Modelos de clasificación
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# XGBoost — usar GradientBoosting si no está instalado
try:
    from xgboost import XGBClassifier
    USE_XGBOOST = True
    print("XGBoost disponible — se usará XGBClassifier.")
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    USE_XGBOOST = False
    print("XGBoost no instalado — usando GradientBoostingClassifier como alternativa.")

# Métricas de evaluación
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

# Persistencia del modelo
import joblib

import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
sns.set_theme(style='whitegrid', color_codes=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Librerías cargadas correctamente.")


---
## 4. Cargar Dataset


In [ ]:
# Cargar el dataset de Débito Directo de Allianz desde el archivo Excel proporcionado.
# El archivo tiene dos hojas: 'Copyright' (metadatos) y 'AllianzDirectDebitData' (los datos reales).
DATA_PATH = 'data/data.xlsx'
SHEET_NAME = 'AllianzDirectDebitData'

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()


---
## 5. Inspección Inicial de Datos


In [ ]:
# Estructura general: tipos de datos y conteos de no nulos
df.info()


In [ ]:
# Conteo y porcentaje de valores faltantes por columna
nulls = pd.DataFrame({
    'Conteo nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(4)
})
print("Valores faltantes por columna:")
nulls


In [ ]:
# Filas duplicadas
n_dupes = df.duplicated().sum()
print(f"Filas duplicadas: {n_dupes:,}")

# Identificadores únicos para entender la granularidad
print(f"Contratos únicos   : {df['Contract_number'].nunique():,}")
print(f"Clientes únicos    : {df['Customer_ID'].nunique():,}")
print(f"Corredores únicos  : {df['Broker_account_number'].nunique():,}")


In [ ]:
# Valores únicos por columna categórica
df.nunique()


In [ ]:
# Estadísticas descriptivas de variables numéricas
df[['Annual_premium', 'Broker_cor']].describe().round(2)


**Observaciones Iniciales:**
- El dataset tiene **452,222 contratos** — un dataset de escala real a gran escala.
- Los valores faltantes son extremadamente bajos (solo 2 filas en todas las columnas, < 0.001%). Probablemente son artefactos de la fila de encabezado.
- Hay **64 filas exactamente duplicadas** que se eliminarán.
- `Broker_cor` tiene un valor máximo extremo (más de 6 × 10¹⁶), lo que indica **valores atípicos de calidad de datos** — esta columna representa un ratio financiero y valores superiores a ~300 son implausibles y deben ser limitados.
- `Annual_premium` tiene una distribución muy sesgada a la derecha (máx ~3M vs. media ~513) — una transformación logarítmica será útil.


---
## 6. Análisis Exploratorio de Datos (EDA)

### 6.1 Distribución de la Variable Objetivo


In [ ]:
# Distribución del objetivo — clasificación binaria
target_counts = df['Is_direct_debit'].value_counts()
target_pct = df['Is_direct_debit'].value_counts(normalize=True).round(4) * 100

print("Distribución de Is_direct_debit:")
print(pd.DataFrame({'Conteo': target_counts, 'Porcentaje (%)': target_pct}))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ['Sin Débito Directo (0)', 'Débito Directo (1)'],
    target_counts.values,
    color=['#d9534f', '#5cb85c'],
    edgecolor='white'
)
for bar, pct in zip(bars, target_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3000,
            f'{pct:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Adopción de Débito Directo — Contratos Allianz', fontsize=13)
ax.set_ylabel('Número de Contratos')
plt.tight_layout()
plt.show()


**Hallazgo clave:** Solo el **21% de los contratos** usa débito directo, frente al ~65% del promedio del mercado. El dataset está **significativamente desbalanceado**. Esto significa que la exactitud por sí sola será una métrica engañosa — debemos priorizar **recall, F1-score y ROC-AUC**.

### 6.2 Débito Directo por Segmento de Cliente


In [ ]:
# Función auxiliar: calcular la tasa de adopción de débito directo por variable categórica
def plot_dd_rate(col, title, figsize=(8, 4)):
    rate = df.groupby(col)['Is_direct_debit'].mean().sort_values(ascending=False)
    count = df.groupby(col)['Is_direct_debit'].count()
    summary = pd.DataFrame({'Tasa DD (%)': (rate * 100).round(2), 'Contratos': count})
    print(summary)
    fig, ax = plt.subplots(figsize=figsize)
    rate.mul(100).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.axhline(21.02, color='red', linestyle='--', label='Promedio Allianz 21%')
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Tasa de Débito Directo (%)')
    ax.set_xlabel(col)
    ax.tick_params(axis='x', rotation=30)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_dd_rate('Customer_segment', 'Tasa de Débito Directo por Segmento de Cliente')


### 6.3 Débito Directo por Frecuencia de Pago


In [ ]:
plot_dd_rate('Payment_frequency', 'Tasa de Débito Directo por Frecuencia de Pago')


**Hallazgo clave:** Los contratos de pago mensual probablemente tienen cerca del 100% de débito directo debido a la regla de negocio de Allianz. Esto confirma el potencial **riesgo de fuga de datos** de `Payment_frequency`.

### 6.4 Débito Directo por Tipo de Cliente


In [ ]:
plot_dd_rate('Customer_type', 'Tasa de Débito Directo por Tipo de Cliente', figsize=(6, 4))


### 6.5 Débito Directo por Región de Cliente y Urbanización


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axs, ['Customer_region', 'Customer_urbanization', 'Broker_urbanization']):
    rate = df.groupby(col)['Is_direct_debit'].mean().sort_values(ascending=False) * 100
    rate.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.axhline(21.02, color='red', linestyle='--', label='Promedio Allianz')
    ax.set_title(f'Tasa DD por {col}', fontsize=11)
    ax.set_ylabel('Tasa de Débito Directo (%)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 6.6 Débito Directo por Región y Provincia del Corredor


In [ ]:
plot_dd_rate('Broker_region', 'Tasa de Débito Directo por Región del Corredor', figsize=(6, 4))
plot_dd_rate('Broker_province', 'Tasa de Débito Directo por Provincia del Corredor', figsize=(10, 4))


### 6.7 Débito Directo por Línea de Negocio y Tipo de Producto


In [ ]:
plot_dd_rate('Line_of_business', 'Tasa de Débito Directo por Línea de Negocio', figsize=(10, 4))


In [ ]:
# Top 20 tipos de producto por tasa de débito directo
top_products = (
    df.groupby('Product_type')['Is_direct_debit']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'Tasa DD', 'count': 'Contratos'})
    .query('Contratos >= 100')
    .sort_values('Tasa DD', ascending=False)
    .head(20)
)
fig, ax = plt.subplots(figsize=(12, 5))
top_products['Tasa DD'].mul(100).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.axhline(21.02, color='red', linestyle='--', label='Promedio Allianz 21%')
ax.set_title('Tasa de Débito Directo por Tipo de Producto (mín 100 contratos)', fontsize=12)
ax.set_ylabel('Tasa de Débito Directo (%)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()


### 6.8 Análisis de Variables Numéricas


In [ ]:
# Distribución de prima anual por estado de débito directo
# Se limita al percentil 99 para evitar que los valores extremos distorsionen la visualización
cap_premium = df['Annual_premium'].quantile(0.99)
df_plot = df[df['Annual_premium'] <= cap_premium]

fig, axs = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(data=df_plot, x='Annual_premium', hue='Is_direct_debit',
             bins=50, multiple='stack', ax=axs[0])
axs[0].set_title('Distribución Prima Anual por Estado DD (limitado al percentil 99)')
axs[0].set_xlabel('Prima Anual (€)')

sns.boxplot(data=df_plot, x='Is_direct_debit', y='Annual_premium', ax=axs[1])
axs[1].set_title('Prima Anual por Estado DD')
axs[1].set_xticklabels(['Sin DD (0)', 'DD (1)'])

plt.tight_layout()
plt.show()


In [ ]:
# Ratio combinado del corredor — limitar valores extremos para visualización
# Un ratio combinado superior a ~300% es implausible para un corredor de seguros
cap_cor = df['Broker_cor'].quantile(0.95)
df_plot_cor = df[df['Broker_cor'] <= cap_cor]

fig, axs = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(data=df_plot_cor, x='Broker_cor', hue='Is_direct_debit',
             bins=50, multiple='stack', ax=axs[0])
axs[0].set_title('Distribución COR del Corredor por Estado DD (limitado al percentil 95)')
axs[0].set_xlabel('Ratio Combinado del Corredor')

sns.boxplot(data=df_plot_cor, x='Is_direct_debit', y='Broker_cor', ax=axs[1])
axs[1].set_title('COR del Corredor por Estado DD')
axs[1].set_xticklabels(['Sin DD (0)', 'DD (1)'])

plt.tight_layout()
plt.show()


### 6.9 Distribución de Edad del Cliente


In [ ]:
plot_dd_rate('Customer_age', 'Tasa de Débito Directo por Grupo de Edad del Cliente', figsize=(9, 4))


### 6.10 Observaciones de Negocio del EDA

Con base en el análisis exploratorio, observamos los siguientes patrones:

- **Frecuencia de pago** es el factor más determinante. Los contratos de pago mensual tienen una adopción de débito directo cercana al 100% debido a la regla de negocio de Allianz — esto crea un riesgo de fuga para los modelos.
- **Segmento de cliente:** Los clientes Retail, PyME y Mediana Corporación muestran tasas de adopción diferentes. Debemos evaluar si tiene sentido dirigir campañas por segmento.
- **Tipo de cliente:** Las personas físicas y las empresas se comportan de manera diferente. Las empresas ('No age') tienden a tener menor adopción de débito directo, probablemente prefiriendo controles de pago manuales.
- **Geografía:** Las regiones y provincias de clientes y corredores muestran variación en la adopción. Las áreas urbanas pueden tener patrones de adopción diferentes a las rurales.
- **Prima anual:** Los contratos con primas más altas pueden o no correlacionar con el débito directo — dejaremos que el modelo determine esto.
- **Broker_cor:** El ratio combinado tiene muchos valores extremos (problema de calidad de datos). Limitaremos estos valores antes del modelado.
- **Línea de negocio:** Algunas líneas de seguros tienen patrones de pago estructuralmente diferentes.


---
## 7. Limpieza de Datos

### 7.1 Eliminar Filas con Todos los Valores Nulos


In [ ]:
# Eliminar filas donde la variable objetivo es nula — no pueden usarse para entrenamiento
print(f"Filas antes de eliminar nulos: {len(df):,}")
df = df.dropna(subset=['Is_direct_debit'])
print(f"Filas después de eliminar nulos: {len(df):,}")

# Convertir objetivo a entero
df['Is_direct_debit'] = df['Is_direct_debit'].astype(int)


### 7.2 Eliminar Registros Duplicados


In [ ]:
# Eliminar filas exactamente duplicadas para evitar sesgo en el entrenamiento
print(f"Filas antes de deduplicar: {len(df):,}")
df = df.drop_duplicates()
print(f"Filas después de deduplicar: {len(df):,}")


### 7.3 Limitar Valores Extremos de Broker_cor

La columna `Broker_cor` tiene valores atípicos extremos (hasta 6 × 10¹⁶). Un ratio combinado superior al 300% no es significativo en un contexto de seguros estándar — estos parecen ser artefactos de calidad de datos. Limitamos al percentil 95 para preservar la variabilidad mientras se eliminan los extremos implausibles.


In [ ]:
# Limitar Broker_cor al percentil 95 para manejar valores atípicos extremos
cor_cap = df['Broker_cor'].quantile(0.95)
print(f"Límite percentil 95 de Broker_cor: {cor_cap:,.2f}")
df['Broker_cor'] = df['Broker_cor'].clip(upper=cor_cap)
print(f"Máximo de Broker_cor tras limitar: {df['Broker_cor'].max():,.2f}")


### 7.4 Manejar Valores Faltantes Restantes


In [ ]:
# Para nulos categóricos restantes: rellenar con 'Unknown'
cat_cols = df.select_dtypes(include='object').columns.tolist()
df[cat_cols] = df[cat_cols].fillna('Unknown')

# Para nulos numéricos: rellenar con la mediana
df['Annual_premium'] = df['Annual_premium'].fillna(df['Annual_premium'].median())
df['Broker_cor'] = df['Broker_cor'].fillna(df['Broker_cor'].median())

print(f"Nulos restantes: {df.isnull().sum().sum()}")


---
## 8. Ingeniería de Características

Creamos características adicionales que pueden ayudar al modelo a capturar patrones relevantes para el negocio.


In [ ]:
# Transformación logarítmica de Annual_premium para reducir sesgo a la derecha
df['Premium_log'] = np.log1p(df['Annual_premium'])

# Indicador binario: cliente empresarial (Customer_type == 'Enterprise')
df['Is_enterprise'] = (df['Customer_type'] == 'Enterprise').astype(int)

# Indicador binario: cliente y corredor en la misma región
df['Customer_Broker_same_region'] = (
    df['Customer_region'] == df['Broker_region']
).astype(int)

# Indicador binario: cliente y corredor en la misma provincia
df['Customer_Broker_same_province'] = (
    df['Customer_province'] == df['Broker_province']
).astype(int)

# Indicador binario: corredor con ratio combinado inferior a 100 (corredor rentable)
df['Broker_profitable'] = (df['Broker_cor'] < 100).astype(int)

# Indicador binario: frecuencia de pago mensual (variable de fuga clave — discutida por separado)
df['Is_monthly_payment'] = (
    df['Payment_frequency'].str.lower() == 'monthly'
).astype(int)

print("Ingeniería de características completada. Nuevas columnas:")
new_cols = ['Premium_log', 'Is_enterprise', 'Customer_Broker_same_region',
            'Customer_Broker_same_province', 'Broker_profitable', 'Is_monthly_payment']
print(df[new_cols].describe().round(3))


---
## 9. Pipeline de Preprocesamiento

### 9.1 Definir Características y Objetivo


In [ ]:
# Variable objetivo
TARGET = 'Is_direct_debit'

# Columnas de identificadores — excluidas del modelado (los IDs únicos causan memorización, no generalización)
ID_COLS = ['Broker_account_number', 'Contract_number', 'Customer_ID']

# Todas las características disponibles para el modelado (Estrategia A: incluye Payment_frequency)
FEATURE_COLS = [
    'Customer_segment', 'Line_of_business', 'Product_type',
    'Annual_premium', 'Premium_log', 'Payment_frequency',
    'Customer_age', 'Customer_type', 'Customer_region', 'Customer_province',
    'Broker_region', 'Broker_province', 'Broker_cor',
    'Customer_urbanization', 'Broker_urbanization',
    'Is_enterprise', 'Customer_Broker_same_region',
    'Customer_Broker_same_province', 'Broker_profitable', 'Is_monthly_payment'
]

# Características Estrategia B — excluir Payment_frequency e Is_monthly_payment
FEATURE_COLS_B = [f for f in FEATURE_COLS
                  if f not in ['Payment_frequency', 'Is_monthly_payment']]

X = df[FEATURE_COLS]
y = df[TARGET]

print(f"Forma de la matriz de características: {X.shape}")
print(f"Distribución del objetivo:")
print(y.value_counts())


In [ ]:
# Separar listas de características categóricas y numéricas
CAT_FEATURES = X.select_dtypes(include='object').columns.tolist()
NUM_FEATURES = X.select_dtypes(exclude='object').columns.tolist()

print(f"Características categóricas ({len(CAT_FEATURES)}): {CAT_FEATURES}")
print(f"Características numéricas   ({len(NUM_FEATURES)}): {NUM_FEATURES}")


In [ ]:
# Pipelines de preprocesamiento:
# - Categóricas: imputar con 'Unknown' y luego codificar one-hot
# - Numéricas: imputar con mediana y luego escalar estándar
#   (escalado necesario para Regresión Logística y SVM; no afecta a modelos de árbol)

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, CAT_FEATURES),
    ('num', numerical_transformer, NUM_FEATURES)
])

print("Pipeline de preprocesamiento definido.")


---
## 10. División Entrenamiento-Prueba


In [ ]:
# División estratificada: preserva la distribución de clases en los conjuntos de entrenamiento y prueba.
# Esto es importante porque el dataset está desbalanceado (~21% débito directo).
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]:,} filas")
print(f"Conjunto de prueba        : {X_test.shape[0]:,} filas")
print(f"\nDistribución del objetivo en entrenamiento:")
print(y_train.value_counts(normalize=True).round(4))
print(f"\nDistribución del objetivo en prueba:")
print(y_test.value_counts(normalize=True).round(4))


---
## 11. Revisión del Desbalance de Clases


In [ ]:
# Medir el ratio de desbalance de clases
n_majority = (y_train == 0).sum()
n_minority = (y_train == 1).sum()
imbalance_ratio = n_majority / n_minority

print(f"Conjunto de entrenamiento — Sin Débito Directo (0): {n_majority:,}")
print(f"Conjunto de entrenamiento — Débito Directo (1)    : {n_minority:,}")
print(f"Ratio de desbalance (0:1)                         : {imbalance_ratio:.2f}:1")

# Estrategia: usar class_weight='balanced' en todos los clasificadores en lugar de sobremuestreo.
# Esto evita la fuga de datos por muestreo antes de la división y es más eficiente
# en un dataset de este tamaño.
# Para XGBoost, usamos scale_pos_weight = mayoría / minoría.
SCALE_POS_WEIGHT = n_majority / n_minority
print(f"\nscale_pos_weight para XGBoost: {SCALE_POS_WEIGHT:.4f}")

print("""
Con un desbalance de ~3.8:1, la exactitud por sí sola será engañosa.
Un modelo ingenuo que siempre predice clase 0 alcanzaría ~79% de exactitud.
Priorizaremos Recall, F1-score y ROC-AUC para la evaluación de modelos.
""")


---
## 12. Modelo 1: Árbol de Decisión

**Justificación:** Los Árboles de Decisión son altamente interpretables. Las divisiones aprendidas pueden traducirse directamente en reglas de negocio — útiles para explicar a la dirección qué características del contrato segmentan a los clientes entre débito directo y no débito directo. Limitamos la profundidad y el mínimo de muestras por hoja para prevenir el sobreajuste.

**Posible debilidad:** Puede sobreajustarse si no se restringe; menos potente que los métodos de conjunto.


In [ ]:
dt_classifier = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=20,
    class_weight='balanced',   # compensa el desbalance 79%/21%
    random_state=42
)

pipeline_dt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', dt_classifier)
])

pipeline_dt.fit(X_train, y_train)
y_pred_dt = pipeline_dt.predict(X_test)
y_prob_dt = pipeline_dt.predict_proba(X_test)[:, 1]

print("=== Árbol de Decisión ===")
print(classification_report(y_test, y_pred_dt, target_names=['Sin DD (0)', 'DD (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_dt):.4f}")


---
## 13. Modelo 2: Random Forest

**Justificación:** Random Forest reduce el sobreajuste al promediar 200 árboles independientes. Típicamente supera a un solo Árbol de Decisión y proporciona puntuaciones de importancia de características que pueden usarse para la interpretación de negocio.

**Posible debilidad:** Menos interpretable que un solo árbol (no hay un único conjunto de reglas); puede ser lento en datasets muy grandes.


In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1   # usar todos los núcleos de CPU
)

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_classifier)
])

pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)
y_prob_rf = pipeline_rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['Sin DD (0)', 'DD (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")


---
## 14. Modelo 3: Regresión Logística

**Justificación:** La Regresión Logística es una línea base sólida y altamente interpretable — los signos y magnitudes de los coeficientes muestran directamente la dirección y fuerza relativa de la relación de cada característica con el débito directo. Es útil para comunicar la lógica del modelo a partes interesadas no técnicas.

**Posible debilidad:** Asume relaciones lineales entre las características y el log-odds; puede tener menor desempeño si la frontera de decisión real es no lineal.


In [ ]:
lr_classifier = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', lr_classifier)
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
y_prob_lr = pipeline_lr.predict_proba(X_test)[:, 1]

print("=== Regresión Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=['Sin DD (0)', 'DD (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")


---
## 15. Modelo 4: Máquina de Soporte Vectorial

**Justificación:** La SVM con kernel RBF puede capturar fronteras de decisión no lineales que los modelos lineales no detectan. Es útil para probar si la complejidad de la frontera de decisión beneficia la predicción.

**Nota práctica:** La SVM-RBF es computacionalmente costosa con más de 360K filas de entrenamiento. Usamos una muestra estratificada de 50,000 filas para hacer el entrenamiento factible, mientras se mantiene el balance de clases. El conjunto de prueba sigue siendo las 90K filas completas para una evaluación justa.

**Posible debilidad:** Menos interpretable; lenta en datasets grandes; requiere entrada escalada.


In [ ]:
# Muestra 50,000 filas estratificadas por objetivo para hacer el entrenamiento de SVM factible
from sklearn.utils import resample as sk_resample

sample_idx = resample(
    X_train.index,
    n_samples=50_000,
    stratify=y_train,
    random_state=42
)
X_train_svm = X_train.loc[sample_idx]
y_train_svm = y_train.loc[sample_idx]

print(f"Subconjunto de entrenamiento SVM: {len(X_train_svm):,} filas")
print(f"Distribución del objetivo en el subconjunto SVM:")
print(y_train_svm.value_counts(normalize=True).round(4))


In [ ]:
svm_classifier = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,  # necesario para calcular predict_proba para ROC-AUC
    random_state=42
)

pipeline_svm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', svm_classifier)
])

pipeline_svm.fit(X_train_svm, y_train_svm)
y_pred_svm = pipeline_svm.predict(X_test)
y_prob_svm = pipeline_svm.predict_proba(X_test)[:, 1]

print("=== Máquina de Soporte Vectorial (RBF, entrenada con muestra de 50K) ===")
print(classification_report(y_test, y_pred_svm, target_names=['Sin DD (0)', 'DD (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_svm):.4f}")


---
## 16. Modelo 5: XGBoost (o Gradient Boosting como alternativa)

**Justificación:** Los modelos de gradient boosting corrigen iterativamente los errores de los árboles anteriores y típicamente logran el mejor desempeño en datos tabulares de negocio. XGBoost es particularmente eficiente con características codificadas en one-hot dispersas.

**Posible debilidad:** Más hiperparámetros que ajustar; menos interpretable sin análisis SHAP; requiere más memoria.


In [ ]:
if USE_XGBOOST:
    boost_classifier = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=SCALE_POS_WEIGHT,  # maneja el desbalance de clases
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    model_name_boost = 'XGBoost'
else:
    from sklearn.ensemble import GradientBoostingClassifier
    boost_classifier = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42
    )
    model_name_boost = 'Gradient Boosting (alternativa a XGBoost)'

pipeline_boost = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', boost_classifier)
])

pipeline_boost.fit(X_train, y_train)
y_pred_boost = pipeline_boost.predict(X_test)
y_prob_boost = pipeline_boost.predict_proba(X_test)[:, 1]

print(f"=== {model_name_boost} ===")
print(classification_report(y_test, y_pred_boost, target_names=['Sin DD (0)', 'DD (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_boost):.4f}")


---
## 17. Comparación de Modelos


In [ ]:
# Compilar tabla de resultados con todas las métricas de evaluación
results = []

for name, y_pred, y_prob in [
    ('Árbol de Decisión',           y_pred_dt,    y_prob_dt),
    ('Random Forest',               y_pred_rf,    y_prob_rf),
    ('Regresión Logística',         y_pred_lr,    y_prob_lr),
    ('Máquina de Soporte Vectorial', y_pred_svm,  y_prob_svm),
    (model_name_boost,              y_pred_boost, y_prob_boost),
]:
    results.append({
        'Modelo': name,
        'Exactitud':  round(accuracy_score(y_test, y_pred), 4),
        'Precisión':  round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':     round(recall_score(y_test, y_pred), 4),
        'F1-Score':   round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':    round(roc_auc_score(y_test, y_prob), 4),
    })

results_df = pd.DataFrame(results).set_index('Modelo')
results_df


In [ ]:
# Comparación visual de F1-Score y ROC-AUC entre modelos
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

results_df['F1-Score'].sort_values().plot(
    kind='barh', ax=axs[0], color='steelblue', edgecolor='white'
)
axs[0].set_title('Comparación de Modelos — F1-Score (clase Débito Directo)', fontsize=12)
axs[0].set_xlabel('F1-Score')
axs[0].set_xlim(0, 1)
for i, v in enumerate(results_df['F1-Score'].sort_values()):
    axs[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

results_df['ROC-AUC'].sort_values().plot(
    kind='barh', ax=axs[1], color='darkorange', edgecolor='white'
)
axs[1].set_title('Comparación de Modelos — ROC-AUC', fontsize=12)
axs[1].set_xlabel('ROC-AUC')
axs[1].set_xlim(0, 1)
for i, v in enumerate(results_df['ROC-AUC'].sort_values()):
    axs[1].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()


---
## 18. Matrices de Confusión


In [ ]:
# Graficar matrices de confusión para los cinco modelos
model_preds = [
    ('Árbol de Decisión',           y_pred_dt),
    ('Random Forest',               y_pred_rf),
    ('Regresión Logística',         y_pred_lr),
    ('Máquina de Soporte Vectorial', y_pred_svm),
    (model_name_boost,              y_pred_boost),
]

fig, axs = plt.subplots(1, 5, figsize=(28, 5))

for ax, (name, y_pred) in zip(axs, model_preds):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Pred Sin DD', 'Pred DD'],
        yticklabels=['Real Sin DD', 'Real DD']
    )
    ax.set_title(name, fontsize=10)

plt.suptitle('Matrices de Confusión — Todos los Modelos', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("""
Interpretación de negocio de las entradas de la matriz de confusión:
  Verdadero Positivo  (abajo-derecha): Contrato predicho DD y es DD. Identificación correcta.
  Verdadero Negativo  (arriba-izq)   : Contrato predicho Sin-DD y es Sin-DD. Correcto.
  Falso Positivo      (arriba-der)   : Predicho DD pero no es DD. Desperdicio de campaña — cliente equivocado.
  Falso Negativo      (abajo-izq)    : Predicho Sin-DD pero es DD. Oportunidad perdida — adopción no capturada.
""")


---
## 19. ROC-AUC y Curvas ROC


In [ ]:
# Curvas ROC para todos los modelos en el mismo gráfico
fig, ax = plt.subplots(figsize=(9, 6))

colors = ['steelblue', 'darkorange', 'green', 'red', 'purple']

for (name, y_pred, y_prob), color in zip(
    [
        ('Árbol de Decisión',           y_pred_dt,    y_prob_dt),
        ('Random Forest',               y_pred_rf,    y_prob_rf),
        ('Regresión Logística',         y_pred_lr,    y_prob_lr),
        ('Máquina de Soporte Vectorial', y_pred_svm,  y_prob_svm),
        (model_name_boost,              y_pred_boost, y_prob_boost),
    ],
    colors
):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, label=f'{name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', label='Línea base aleatoria (AUC = 0.50)')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)')
ax.set_title('Curvas ROC — Predicción de Débito Directo Allianz', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()


---
## 20. Importancia de Características e Interpretación


In [ ]:
# Extraer nombres de características tras la codificación one-hot
ohe_feature_names = (
    pipeline_rf.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(CAT_FEATURES)
).tolist()
all_feature_names = ohe_feature_names + NUM_FEATURES

# --- Importancia de características Random Forest ---
rf_imp = pd.Series(
    pipeline_rf.named_steps['classifier'].feature_importances_,
    index=all_feature_names
).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
rf_imp.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Random Forest — Top 20 Importancias de Características', fontsize=13)
ax.set_xlabel('Puntuación de Importancia')
plt.tight_layout()
plt.show()

print("Top 20 características (Random Forest):")
print(rf_imp.reset_index().rename(columns={'index': 'Característica', 0: 'Importancia'}))


In [ ]:
# --- Importancia de características Árbol de Decisión ---
dt_imp = pd.Series(
    pipeline_dt.named_steps['classifier'].feature_importances_,
    index=all_feature_names
).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
dt_imp.sort_values().plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
ax.set_title('Árbol de Decisión — Top 20 Importancias de Características', fontsize=13)
ax.set_xlabel('Puntuación de Importancia')
plt.tight_layout()
plt.show()


**Interpretación de negocio de las características clave:**

- **`Payment_frequency_Monthly` / `Is_monthly_payment`:** Si esta es la característica principal, refleja la regla de negocio de Allianz de que los contratos mensuales deben usar débito directo. Esto es esperado pero constituye una **fuga casi total** — no nos dice nada sobre el comportamiento de adopción voluntaria.
- **`Customer_segment`:** Si Retail vs. PyME vs. Mediana Corporación aparecen entre las características principales, indica que las estrategias de campaña deben diferenciarse por segmento.
- **`Customer_type`:** Empresa vs. Persona física — los clientes empresariales típicamente usan facturación y resisten el débito directo. Las personas físicas son mejores objetivos de conversión.
- **`Line_of_business` / `Product_type`:** Ciertos productos de seguro (p.ej., automóvil, edificio) pueden tener incentivos o convenciones estructurales que lleven a mayor adopción de débito directo.
- **`Annual_premium` / `Premium_log`:** El tamaño de la prima puede indicar qué tan deliberado es el proceso de pago. Los clientes con primas altas pueden preferir el control manual.
- **`Broker_cor` / `Broker_profitable`:** La rentabilidad del corredor se correlaciona con su calidad. Los corredores de alto rendimiento pueden ser más proactivos en recomendar el débito directo.
- **`Customer_region` / `Customer_urbanization`:** Los factores geográficos pueden reflejar diferencias culturales o de infraestructura bancaria en Bélgica.


---
## 20b. Estrategia B — Modelo Sin Frecuencia de Pago (Análisis de Fuga)


In [ ]:
# Estrategia B: excluir Payment_frequency e Is_monthly_payment
# Esto revela qué señales conductuales y demográficas impulsan el débito directo
# más allá de la regla mensual obligatoria.

X_B = df[FEATURE_COLS_B]
y_B = df[TARGET]

CAT_B = X_B.select_dtypes(include='object').columns.tolist()
NUM_B = X_B.select_dtypes(exclude='object').columns.tolist()

preprocessor_B = ColumnTransformer(transformers=[
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), CAT_B),
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler())
    ]), NUM_B)
])

X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
    X_B, y_B, test_size=0.2, random_state=42, stratify=y_B
)

# Usar Random Forest para la comparación de Estrategia B
pipeline_rf_B = Pipeline([
    ('preprocessor', preprocessor_B),
    ('classifier', RandomForestClassifier(
        n_estimators=200, min_samples_leaf=10,
        class_weight='balanced', random_state=42, n_jobs=-1
    ))
])

pipeline_rf_B.fit(X_train_B, y_train_B)
y_pred_rf_B = pipeline_rf_B.predict(X_test_B)
y_prob_rf_B = pipeline_rf_B.predict_proba(X_test_B)[:, 1]

print("=== Estrategia B — Random Forest SIN Payment_frequency ===")
print(classification_report(y_test_B, y_pred_rf_B, target_names=['Sin DD (0)', 'DD (1)']))
print(f"ROC-AUC: {roc_auc_score(y_test_B, y_prob_rf_B):.4f}")

print(f"\nROC-AUC Estrategia A (con Payment_frequency)   : {roc_auc_score(y_test, y_prob_rf):.4f}")
print(f"ROC-AUC Estrategia B (sin Payment_frequency)    : {roc_auc_score(y_test_B, y_prob_rf_B):.4f}")
print("""
Si el ROC-AUC de Estrategia B es significativamente menor, confirma que Payment_frequency
domina el modelo y que las señales conductuales/demográficas por sí solas tienen menor poder predictivo.
Ambos modelos tienen valor de negocio: Estrategia A para puntuación, Estrategia B para entender los factores conductuales.
""")


---
## 21. Selección del Modelo Final


In [ ]:
# Mostrar tabla de comparación final para la selección del modelo
print("=== Resumen de Selección de Modelo ===")
print(results_df.sort_values('ROC-AUC', ascending=False))


### Lógica de Selección del Modelo

Utilizamos el siguiente marco de decisión:

| Criterio | Peso | Por qué |
|---|---|---|
| ROC-AUC | Alto | Mejor capacidad de clasificación general — importante para puntuar todos los contratos |
| F1-Score | Alto | Equilibra precisión y recall bajo desbalance de clases |
| Recall | Medio | Perder un cliente listo para débito directo es una oportunidad perdida |
| Interpretabilidad | Medio | La dirección y los corredores necesitan entender los factores determinantes |
| Facilidad de despliegue | Bajo | Todos los pipelines de sklearn son sencillos de desplegar |

**Guía de selección:**
- Si **Random Forest o XGBoost** logra el mejor ROC-AUC y F1, recomendarlo como el **modelo de puntuación operativo**.
- Si **Árbol de Decisión** está cerca en rendimiento, recomendarlo como el **modelo de explicabilidad** para talleres de negocio y capacitación de corredores.
- Si **Regresión Logística** es competitiva, preferirla por simplicidad y facilidad de explicación a equipos de finanzas y marketing.
- **SVM** se entrena sobre una submuestra y es menos competitiva en un dataset de esta escala — útil como referencia, no recomendada para producción.

El modelo final seleccionado debe guardarse con Joblib para su despliegue.


In [ ]:
# Identificar el mejor modelo por ROC-AUC
best_model_name = results_df['ROC-AUC'].idxmax()
best_roc = results_df.loc[best_model_name, 'ROC-AUC']
print(f"Mejor modelo por ROC-AUC: {best_model_name} ({best_roc:.4f})")

# Mapear nombre al pipeline
model_pipelines = {
    'Árbol de Decisión': pipeline_dt,
    'Random Forest': pipeline_rf,
    'Regresión Logística': pipeline_lr,
    'Máquina de Soporte Vectorial': pipeline_svm,
    model_name_boost: pipeline_boost,
}
best_pipeline = model_pipelines[best_model_name]


---
## 22. Recomendaciones de Negocio

### Recomendación 1: Puntuación Prioritaria — Dirigirse Primero a los Mejores Candidatos


In [ ]:
# Puntuar todos los contratos SIN débito directo y ordenar por probabilidad predicha
non_dd_df = df[df['Is_direct_debit'] == 0].copy()

# Predecir la probabilidad de débito directo usando el mejor pipeline
non_dd_df['probabilidad_debito_directo'] = best_pipeline.predict_proba(
    non_dd_df[FEATURE_COLS]
)[:, 1]

# Ordenar y mostrar los 10 candidatos con mayor probabilidad
top_targets = (
    non_dd_df[[
        'Contract_number', 'Customer_segment', 'Line_of_business',
        'Customer_type', 'Customer_region', 'Annual_premium',
        'Payment_frequency', 'probabilidad_debito_directo'
    ]]
    .sort_values('probabilidad_debito_directo', ascending=False)
    .head(10)
)

print("Top 10 contratos con mayor probabilidad de convertirse a débito directo:")
top_targets


In [ ]:
# Distribución de probabilidades de todos los contratos sin DD
fig, ax = plt.subplots(figsize=(9, 4))
non_dd_df['probabilidad_debito_directo'].hist(
    bins=50, ax=ax, color='steelblue', edgecolor='white'
)
ax.axvline(0.5, color='red', linestyle='--', label='Umbral 0.5')
ax.set_title('Probabilidad Predicha de Débito Directo — Contratos Sin DD', fontsize=12)
ax.set_xlabel('Probabilidad Predicha de Débito Directo')
ax.set_ylabel('Número de Contratos')
ax.legend()
plt.tight_layout()
plt.show()

# Contar contratos por encima del umbral del 50%
n_high_prob = (non_dd_df['probabilidad_debito_directo'] >= 0.5).sum()
print(f"\nContratos con probabilidad predicha >= 50%: {n_high_prob:,}")
print("Estos representan los objetivos prioritarios para la campaña de débito directo de Allianz.")


### Recomendación 2: Estrategia de Campaña por Segmento

Con base en el EDA y el análisis de importancia de características, Allianz debe diseñar campañas diferenciadas:

- Las **personas físicas Retail** con frecuencias de pago no mensuales son el objetivo de conversión más natural — ya están cómodas con pagos automáticos para gastos personales.
- Los clientes **PyME y Mediana Corporación** requieren un enfoque diferente — típicamente, estos clientes son gestionados por gerentes de cuenta dedicados o corredores. Las conversaciones de corredor sobre la facilidad del débito directo pueden ser más efectivas que el contacto directo al cliente.
- Las líneas de **seguro de automóvil y edificio** pueden tener ventajas estructurales para la adopción de débito directo debido a la regularidad de las renovaciones.
- Los contratos de **pago trimestral** son los siguientes candidatos más probables para frecuencia mensual — estos clientes ya aceptan alguna forma de pago periódico automático.

### Recomendación 3: Trabajar con los Corredores

Si las variables relacionadas con el corredor aparecen entre las características principales:
- Compartir **tableros de control por corredor** que muestren la tasa de conversión de débito directo de cada corredor frente al benchmark del mercado.
- Capacitar a los corredores sobre cómo comunicar el valor del débito directo a los clientes en la renovación de pólizas.
- Introducir **incentivos para corredores** (p.ej., bonos de calidad de servicio) vinculados a tasas de adopción de débito directo.
- Identificar corredores con bajos ratios combinados (portafolios rentables y bien gestionados) y trabajar con ellos como promotores.

### Recomendación 4: Validar la Regla de Pago Mensual

El modelo de Estrategia B (sin frecuencia de pago) revela los factores conductuales y demográficos de la adopción voluntaria de débito directo. Allianz debe:
- Usar la **Estrategia A** (modelo completo) para puntuación operativa y priorización de campañas.
- Usar la **Estrategia B** (modelo sin fuga) para planificación estratégica y comprensión de lo que impulsa la adopción más allá de las reglas de negocio.

### Recomendación 5: Campaña Piloto con Pruebas Controladas


In [ ]:
# Diseño del piloto: top 5,000 contratos por probabilidad vs. grupo de control aleatorio
top_5000 = non_dd_df.nlargest(5000, 'probabilidad_debito_directo')
avg_prob_top = top_5000['probabilidad_debito_directo'].mean()
avg_prob_overall = non_dd_df['probabilidad_debito_directo'].mean()

print("=== Diseño de Campaña Piloto ===")
print(f"Grupo objetivo (top 5,000)  — probabilidad promedio predicha: {avg_prob_top:.2%}")
print(f"Línea base (todos sin DD)    — probabilidad promedio predicha: {avg_prob_overall:.2%}")
print()
print("Diseño del piloto:")
print("  - Contactar los top 5,000 contratos ordenados por probabilidad del modelo")
print("  - NO contactar un grupo de control aleatorio de 5,000 contratos similares")
print("  - Después de 90 días, comparar tasas de conversión a débito directo")
print("  - Usar los resultados para calibrar el umbral del modelo y reentrenar")


---
## 23. Limitaciones y Próximos Pasos

### Limitaciones

1. **Fuga de frecuencia de pago:** La correlación casi perfecta entre pago mensual y débito directo refleja una regla de negocio, no un patrón aprendible. El modelo parecerá altamente preciso en el conjunto de prueba en parte debido a esta regla.

2. **Instantánea del dataset:** El dataset representa una instantánea estática de contratos. El comportamiento del cliente, la combinación de productos y el portafolio del corredor pueden cambiar con el tiempo. El modelo debe reentrenarse periódicamente.

3. **Sin historial de respuesta a campañas:** Predecimos quién tiene *probabilidad de usar ya* o *ser convertido a* débito directo basándose en características del contrato — no en datos reales de respuesta a campañas. Un verdadero modelo de uplift requeriría resultados de pruebas A/B.

4. **Calidad de datos en Broker_cor:** Los valores extremos en `Broker_cor` (hasta 6 × 10¹⁶) indican posibles errores de extracción o codificación de datos. La estrategia de limitación es una solución pragmática pero debe investigarse más con el equipo de ingeniería de datos.

5. **Escalabilidad de SVM:** La SVM se entrenó con una submuestra de 50,000 filas debido a restricciones computacionales en el conjunto de entrenamiento completo de 360K+ filas. Su evaluación puede no representar completamente su rendimiento a escala.

### Próximos Pasos

1. **Análisis SHAP:** Aplicar SHAP (SHapley Additive exPlanations) al mejor modelo para producir explicaciones a nivel individual — útiles para conversaciones con corredores.
2. **Modelado de uplift:** Si se dispone de datos de campañas, construir un verdadero modelo causal que prediga la adopción *incremental* (clientes que NO habrían adoptado sin el alcance).
3. **Ajuste de hiperparámetros:** Aplicar GridSearchCV u Optuna para optimizar aún más el mejor modelo.
4. **Despliegue en API:** Empaquetar el mejor pipeline usando FastAPI o una plataforma de ML en la nube (Azure ML, AWS SageMaker) para puntuación operativa.
5. **Cadencia de reentrenamiento:** Programar el reentrenamiento trimestral del modelo a medida que se acumulen nuevos datos de contratos.


---
## 24. Conclusiones Finales

Este proyecto demuestra cómo Allianz puede usar machine learning supervisado para identificar contratos y perfiles de clientes con mayor probabilidad de usar o ser convertidos a débito directo.

Al entrenar y comparar cinco modelos de clasificación — Árbol de Decisión, Random Forest, Regresión Logística, Máquina de Soporte Vectorial y XGBoost — en un dataset real de **452,222 contratos de seguro**, hemos construido un sistema de puntuación basado en datos que puede:

- Ordenar los contratos sin débito directo por su probabilidad de conversión.
- Identificar las características de cliente, corredor, producto y geografía más asociadas con la adopción de débito directo.
- Apoyar campañas priorizadas y rentables dirigidas a los candidatos más probables.

El modelo cierra la brecha entre la tasa de adopción del 21% de Allianz y el promedio del mercado del 65% mediante un proceso analítico estructurado, repetible y auditable alineado con la metodología CRISP-DM.

Al priorizar la adopción de débito directo mediante campañas basadas en datos, Allianz puede:
- **Mejorar la predictibilidad del flujo de caja** para el equipo de finanzas.
- **Reducir los costos de cobro manual** para el equipo de operaciones.
- **Acortar el ciclo de cuentas por cobrar** para la tesorería y la gestión del capital de trabajo.
- **Empoderar a los corredores** con información para tener conversaciones más productivas con los clientes.
- **Ofrecer una mejor experiencia** a los clientes mediante pagos automáticos y puntuales de primas.

> *Analizamos el portafolio de contratos de Allianz con machine learning. Seleccionamos el modelo con el mejor equilibrio entre rendimiento e interpretabilidad. Recomendamos desplegarlo como una herramienta de puntuación operativa, comenzando con una campaña piloto controlada para validar su impacto real de conversión.*


---
## 25. Guardar Mejor Modelo con Joblib


In [ ]:
# Guardar el mejor pipeline (preprocesador + clasificador) en disco
# El objeto pipeline contiene todo lo necesario para puntuar nuevos datos
OUTPUT_PATH = f'modelo_{best_model_name.replace(" ", "_")}_allianz_dd.pkl'

joblib.dump(best_pipeline, OUTPUT_PATH)
print(f"Mejor modelo guardado en: {OUTPUT_PATH}")

# También guardar la tabla completa de resultados para el informe
results_df.to_csv('allianz_comparacion_modelos.csv')
print("Tabla de comparación de modelos guardada en: allianz_comparacion_modelos.csv")

print()
print("Para cargar y puntuar nuevos contratos:")
print(f"  pipeline = joblib.load('{OUTPUT_PATH}')")
print(f"  probabilidades = pipeline.predict_proba(nuevos_contratos[FEATURE_COLS])[:, 1]")
